# Week 1 Lecture 1 - Wet world

Throughout the course we will be using these [Jupyter notebooks](https://jupyter.org/) to develop, run, and share code. This version of the notebook uses R, with [Stan](https://mc-stan.org/) (via the [`cmdstanr`](https://mc-stan.org/cmdstanr/) package) for model fitting.

McElreath's lectures for the whole book are available here: https://github.com/rmcelreath/stat_rethinking_2022

An R/Stan repo of code is available here: https://vincentarelbundock.github.io/rethinking2/

An excellent port to Python/PyMC Code is available here: https://github.com/dustinstansbury/statistical-rethinking-2023

You are encouraged to work through both of these versions to re-enforce what we're doing in class.

**Setup (once):** these notebooks use the `cmdstanr`, `posterior`, `bayesplot`, and `dagitty` packages, plus CmdStan itself:

```r
install.packages(c("cmdstanr", "posterior", "bayesplot"), repos = c("https://stan-dev.r-universe.dev", getOption("repos")))
install.packages("dagitty")
cmdstanr::check_cmdstan_toolchain(fix = TRUE)
cmdstanr::install_cmdstan()
```

Stan compiles each model to C++, so you need a working C++ compiler (on a Mac, the Xcode command line tools: `xcode-select --install`).

In [ ]:
# Load R packages
library(cmdstanr)    # R interface to Stan
library(posterior)   # Working with posterior draws

# Grid approximation

The best way to understand how Bayes theorem works is to work through it by hand - the grid approximation is a good way to do this because you can see visually how parameters and data interact through the likelhood. It also gives a sense of how priors become posteiors.

We'll use the globe tossing example because it conveys so well how probability can be built up and used from scratch.

In [ ]:
# Define grid
p_grid <- seq(0, 1, length.out = 20)
p_grid

In [ ]:
# Define prior
prior <- rep(1, 20)
prior

In [ ]:
# Normalize
prior <- prior/sum(prior)
prior1 <- prior
prior

Next we need to define the likelihood which, because it is descrete, is given by the probability mass function (PMF)

$$
\binom{n}{k}p^{x}(1-p)^{n-x}
$$

where

$$
\binom{n}{k} = \frac{n!}{k!(n - k)!}
$$

which in code is

In [ ]:
# Binomial distribution
my_dbinom <- function(x, n, p){
    factorial(n)/(factorial(x)*factorial(n-x))*p^x*(1-p)^(n-x)
}

R also has the binomial PMF built in as `dbinom(x, size, prob)`, so we can check our hand-built version against it:

In [ ]:
my_dbinom(3, 10, 0.5); dbinom(3, 10, 0.5)

# Data loop

In [ ]:
# Plot prior over range of p_grid
plot(p_grid, prior, type = "l", xlab = "Proportion water", ylab = "Plausability")

In [ ]:
# New observations
W <- 1
L <- 0
# Number of trials
N <- W+L

# Calculate likelihood
likelihood <- my_dbinom(W, N, p_grid)
likelihood

In [ ]:
# Plot likelihood over range of p_grid
plot(p_grid, likelihood, type = "l", xlab = "Proportion water", ylab = "Likelihood")

In [ ]:
sum(likelihood)

In [ ]:
likelihood*prior

In [ ]:
# Normalizing constant
sum(likelihood*prior)

In [ ]:
# Bayes theorem
posterior <- (likelihood*prior)/sum(likelihood*prior)
posterior

In [ ]:
# Plot posterior over range of p_grid
plot(p_grid, posterior, type = "l", xlab = "Proportion water", ylab = "Posterior",
     ylim = range(c(posterior, prior)))
lines(p_grid, prior, lty = 3)

In [ ]:
# Update new prior
prior <- posterior

Or, we can do it all at once, with all the data

In [ ]:
# New observations
W <- 8
L <- 5
# Number of trials
N <- W+L

# Calculate likelihood
likelihood <- my_dbinom(W, N, p_grid)
# Bayes theorem
posterior <- (likelihood*prior1)/sum(likelihood*prior1)
posterior

In [ ]:
# Plot posterior over range of p_grid
plot(p_grid, posterior, type = "l", xlab = "Proportion water", ylab = "Posterior")
lines(p_grid, prior1, lty = 3)

With this code we have built a Bayesian model to estimate the proportion of water on the earth's surface by recording the number of times our right hand lands on water versus land, given a default 'ignorant' prior. However our heterogenious educations give us some sense better than ignorance about what proportion of the earth is water. We can encode this information in a new prior and see what the data show. There are many ways to do this, but one way is to weight each value of p_grid by our sense of how likely they are.

In [ ]:
p_grid

In [ ]:
# Subjective prior
my_prior <- c(0,0,0,0,0,0,0,0,0.02,0.05,0.10,0.15,0.20,0.20,0.15,0.10,0.05,0.02,0,0)

In [ ]:
sum(my_prior)

In [ ]:
# Standardize
my_prior <- my_prior/sum(my_prior)

In [ ]:
sum(my_prior)

In [ ]:
# Plot my prior over range of p_grid
plot(p_grid, my_prior, type = "l", xlab = "Proportion water", ylab = "Prior")

In [ ]:
# Bayes theorem
posterior2 <- (likelihood*my_prior)/sum(likelihood*my_prior)
posterior2

In [ ]:
# Plot posterior2 over range of p_grid
plot(p_grid, rep(0.05, 20), type = "l", col = "dodgerblue", lty = 3,
     ylim = range(c(my_prior, posterior, posterior2)),
     xlab = "Proportion water", ylab = "Posterior")
lines(p_grid, my_prior, col = "red", lty = 3)
lines(p_grid, posterior, col = "dodgerblue")
lines(p_grid, posterior2, col = "red")
legend("topleft", legend = c("Flat prior", "My prior", "Flat posterior", "My posterior"),
       col = c("dodgerblue", "red", "dodgerblue", "red"), lty = c(3, 3, 1, 1), bty = "n")

# Quadratic approximation

Grid approximations are helpful for learning how Bayes theorem works but they do not scale well, exponentially in fact relative to the number of parameters in the model. For most models we'll use Markov Chain Monte Carlo (MCMC) methods, but before that the quadratic approximation (QA) is a useful method to learn as well as it is really fast and forms the basis of an important MCMC alternative, the [Laplace approximation](https://bookdown.org/rdpeng/advstatcomp/laplace-approximation.html), which is used in [INLA](http://www.r-inla.org/) (with a gentle intro here: https://www.precision-analytics.ca/blog/a-gentle-inla-tutorial/).

McElreath has written his own quadratic approximation algorithm `quap()` for finding the normal peak and standard deviation and you can see that on pg. 42 of the book. In Stan we can use the `$optimize()` method to do the same thing (`quap` uses the same idea under the hood), which finds the MAP. MAP stands for maximum *a posteriori*, and reflects the fact that the algorithm estimates the mode of a posterior, rather than just the likelihood alone. Is is basically a fast optimization algorhithm.

First we write the model in the Stan language. Every Stan program has blocks: `data` (what we observe), `parameters` (what we want to learn about), and `model` (priors and likelihood):

In [ ]:
# Globe tossing model in Stan
globe_code <- "
data {
  int<lower=0> N;   // number of tosses
  int<lower=0> W;   // number of waters
}
parameters {
  real<lower=0, upper=1> p;
}
model {
  p ~ uniform(0, 1);
  W ~ binomial(N, p);
}
"
# Compile the model
globe_qa <- cmdstan_model(write_stan_file(globe_code))

In [ ]:
# Find the posterior mode (MAP)
map_fit <- globe_qa$optimize(data = list(N = N, W = W), refresh = 0)
map_fit$mle()

The MAP gives us the peak of the posterior, but for a quadratic approximation we also need its curvature to get a standard deviation. That comes from the Hessian:

In [ ]:
# Mean of normal approximation
mean_q <- map_fit$mle("p")
mean_q

In [ ]:
# Log-posterior (un-normalized) as an R function
log_post <- function(p) dbinom(W, N, p, log = TRUE) + dunif(p, 0, 1, log = TRUE)
# Hessian (second derivative) of the log posterior at the MAP
H <- optimHess(mean_q, function(p) -log_post(p))
# Standard deviation of normal approximation
std_q <- sqrt(1/H[1, 1])
std_q

One oddity in the code above is the Hessian (`optimHess`) statement, if you've not seen a Hessian before, it is simply (!) a matrix of second-order partial derivatives that can be used to calculate variance/covariance.

Stan can also do the whole Laplace approximation for us with the `$laplace()` method, which draws samples from a normal approximation centred at the mode (note Stan builds this normal on the unconstrained, log-odds scale for `p`, so it is close to, but not exactly, our hand-built version):

In [ ]:
laplace_fit <- globe_qa$laplace(data = list(N = N, W = W), mode = map_fit, jacobian = FALSE, refresh = 0)
laplace_fit$summary("p")

In [ ]:
# analytical calculation
y1 <- my_dbinom(W, N, p_grid)
# quadratic approximation
y2 <- dnorm(p_grid, mean_q, std_q)

plot(p_grid, y1/sum(y1), type = "l", col = "dodgerblue",
     ylim = range(c(y1/sum(y1), y2/sum(y2))),
     main = "n = 13", xlab = "Proportion water", ylab = "Density")
lines(p_grid, y2/sum(y2), col = "orange")
legend("topleft", legend = c("True posterior", "Quadratic approximation"),
       col = c("dodgerblue", "orange"), lty = 1, bty = "n")